# 08. RAG・API を作る

このノートは、[colab-oss-lab](https://github.com/moruku36/colab-oss-lab) の実験 08 です。

[実験06](../docs/results/06_qlora.md) では、LoRA で知識を覚えさせても、聞き方を変えると **44%** しか答えられませんでした。
ここでは同じ質問に **RAG（資料を検索して、その資料を見ながら答える）** で挑みます。モデルは **OpenAI 互換の API サーバー**として動かします。

- モデル: `cyankiwi/gemma-4-26B-A4B-it-AWQ-4bit`（[実験04](../docs/results/04_vllm_speedup.md) のおすすめ）を **vLLM の API サーバー**で動かす
- 資料: このリポジトリの README・解説ページ・実験記録（GitHub から取得して、小さな段落に分ける）
  - ただし 06 の記録（学習前後の答えの例に **間違った答え** が載っている）は資料から外す
- 検索: `intfloat/multilingual-e5-small`（日本語対応の小さな埋め込みモデル、CPU で動かす）
- 呼び出し: `openai` ライブラリ（ChatGPT の API と同じ書き方）で `http://localhost:8000/v1` を呼ぶ

測るもの:

1. 06 と同じ 25 問（学習に使っていない聞き方）の正答率: **RAG なし / RAG あり**（06 の QLoRA は 44%）
2. 検索が正解の書かれた段落を拾えたか
3. **資料に答えがない質問**（5 問）で「記載がありません」と言えるか（作り話をしないか）
4. API の速さ: 1 件ずつ順番に聞く / 25 件を同時に聞く

「想定どおり」とは:

- RAG なしは 20% 以下（モデルは元々知らない）
- **RAG ありは 80% 以上**（06 の QLoRA の 44% を大きく上回る）
- 検索が正解の段落を拾えた割合が 80% 以上
- 資料にない質問 5 問のうち 4 問以上で「記載がありません」と答える
- 同時に聞くと、順番に聞くより合計時間が短い

所要時間の目安: 20〜30 分。

---

## 実行する前に

1. **ランタイム → ランタイムのタイプを変更 → L4 GPU → Save**
2. 上から順に ▶（または「すべてのセルを実行」）
3. 終わったら **ランタイム → セッションを管理 → 解放**

## 1. GPU を確認して、ライブラリを入れる

In [ ]:
# ノート本体では GPU を使わない（GPU は vLLM の API サーバーが使う。検索は CPU）
import subprocess
q = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True)
gpu_name, mem_mib = [x.strip() for x in q.strip().split(",")]
vram_total_gb = int(mem_mib) / 1024
assert "L4" in gpu_name, f"GPU が L4 ではありません: {gpu_name}"
print("GPU:", gpu_name, round(vram_total_gb, 1), "GB")

In [ ]:
# vLLM は専用の仮想環境に入れる（実験04 と同じ。Colab の torchaudio とぶつかるため）
!pip install -q uv
!uv venv -q --allow-existing /content/vllm-env
!uv pip install --python /content/vllm-env/bin/python vllm 2>&1 | tail -2
# ノート本体（検索と API の呼び出し）用
!pip install -q -U openai sentence-transformers 2>&1 | tail -2
!/content/vllm-env/bin/python -c "import vllm; print('vllm', vllm.__version__)"

## 2. vLLM を OpenAI 互換の API サーバーとして起動する

裏でサーバーを動かし、`http://localhost:8000/v1` で待ち受けます。準備ができるまで数分かかります。

- `--language-model-only`: 画像の部品を読み込まない
- `--max-model-len 8192`: 資料を入れても足りる長さ
- `--kv-cache-dtype fp8`: 会話の記憶を小さくする
- 仮想環境の `bin` を `PATH` に入れて起動する（入れないと、起動の途中で `ninja` が見つからずに止まる）


In [ ]:
import os, re, time, requests

MODEL = "cyankiwi/gemma-4-26B-A4B-it-AWQ-4bit"
server_log = open("/content/vllm_server.log", "w")
t0 = time.time()
server = subprocess.Popen(
    ["/content/vllm-env/bin/vllm", "serve", MODEL,
     "--served-model-name", "gemma4-26b",
     "--language-model-only",
     "--max-model-len", "8192",
     "--gpu-memory-utilization", "0.92",
     "--kv-cache-dtype", "fp8",
     "--max-num-batched-tokens", "2048",
     "--max-num-seqs", "16",
     "--port", "8000"],
    stdout=server_log, stderr=subprocess.STDOUT,
    # 仮想環境の bin を PATH に入れる（vLLM が起動中に ninja などの道具を呼ぶため。入れないと FileNotFoundError: 'ninja'）
    env={**os.environ, "PATH": "/content/vllm-env/bin:" + os.environ["PATH"]})

while True:
    if server.poll() is not None:
        causes = [l for l in open("/content/vllm_server.log").read().splitlines()
                  if re.search(r"(Error|error:|out of memory)", l) and "File " not in l]
        print("\n".join(causes[-8:]))
        raise RuntimeError("vLLM サーバーが止まりました（上の原因の行、または /content/vllm_server.log を確認）")
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(5)
server_start_min = (time.time() - t0) / 60
print(f"サーバー起動: {server_start_min:.1f} 分")
print(requests.get("http://localhost:8000/v1/models").json()["data"][0]["id"])
!grep -E "GPU KV cache size|Model loading took" /content/vllm_server.log | tail -2

### API を呼んでみる

ChatGPT の API と同じ書き方で呼べます。`base_url` を自分のサーバーに向けるだけです。

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")  # 鍵は使わない（自分の Colab の中だけ）
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

r = client.chat.completions.create(
    model="gemma4-26b",
    messages=[{"role": "user", "content": "小学校の児童にも分かる言葉で、GPUとVRAMの違いを3文で説明してください。"}],
    max_tokens=200, temperature=0, extra_body=NO_THINK)
print(r.choices[0].message.content)
print(r.usage)

## 3. 資料を集めて、検索できるようにする

GitHub から README・解説ページ・実験記録を取ってきて、見出しごと・約 600 文字ごとの段落に分けます。
それぞれを埋め込みモデルで数字の並び（ベクトル）にして、質問に近い段落を探せるようにします。

In [ ]:
import re, json, urllib.request
import numpy as np
from sentence_transformers import SentenceTransformer

RAW = "https://raw.githubusercontent.com/moruku36/colab-oss-lab/main/"
FILES = ["README.md",
         "docs/01-what-is-colab.md", "docs/02-google-ai-pro.md", "docs/03-what-you-can-do.md",
         "docs/04-oss-models.md", "docs/05-gpu-basics.md", "docs/06-quantization.md", "docs/glossary.md",
         "docs/results/01_first_run.md", "docs/results/02_quantization_max.md", "docs/results/03_thinking_on_off.md",
         "docs/results/04_vllm_speedup.md", "docs/results/05_quantization_compare.md", "docs/results/07_l4_limit.md"]
# 06_qlora.md は外す（学習後の「間違った答え」の例がたくさん載っているため）

def chunks_of(path, text, size=600):
    out, head, buf = [], "", ""
    for line in text.splitlines():
        if line.startswith("#"):
            if buf.strip():
                out.append((path, head, buf.strip()))
            head, buf = line.lstrip("# ").strip(), ""
            continue
        if len(buf) + len(line) > size and buf.strip():
            out.append((path, head, buf.strip()))
            buf = ""
        buf += line + "\n"
    if buf.strip():
        out.append((path, head, buf.strip()))
    return out

chunks = []
for f in FILES:
    text = urllib.request.urlopen(RAW + f).read().decode("utf-8")
    chunks += chunks_of(f, text)
print("段落の数:", len(chunks))

embedder = SentenceTransformer("intfloat/multilingual-e5-small", device="cpu")
t = time.time()
passages = [f"passage: {p} / {h}\n{b}" for p, h, b in chunks]
P = embedder.encode(passages, batch_size=32, normalize_embeddings=True, show_progress_bar=False)
print(f"埋め込み: {time.time() - t:.1f} 秒 / 形 {P.shape}")

def search(q, k=5):
    v = embedder.encode([f"query: {q}"], normalize_embeddings=True)[0]
    idx = np.argsort(-(P @ v))[:k]
    return [chunks[i] for i in idx]

for p, h, b in search("vLLM で 31B を動かしたときの速さ", k=3):
    print("-", p, "/", h, "/", b[:60].replace("\n", " "))

## 4. RAG で答える関数と、評価する質問

In [ ]:
SYSTEM_RAG = ("あなたは colab-oss-lab（Google Colab で OSS の AI モデルを試すリポジトリ）の案内役です。"
              "下の【資料】だけを根拠に、日本語で短く答えてください。"
              "資料に答えが書かれていなければ、推測せずに「資料に記載がありません」と答えてください。"
              "最後に（出典: ファイル名）を付けてください。")
SYSTEM_PLAIN = "You are a helpful assistant. Answer in Japanese. 短く答えてください。"

def ask(q, rag=True, k=5):
    if rag:
        ctx = search(q, k)
        docs = "\n\n".join(f"[{p} / {h}]\n{b}" for p, h, b in ctx)
        msgs = [{"role": "system", "content": SYSTEM_RAG},
                {"role": "user", "content": f"【資料】\n{docs}\n\n【質問】\n{q}"}]
    else:
        ctx = []
        msgs = [{"role": "system", "content": SYSTEM_PLAIN}, {"role": "user", "content": q}]
    t = time.time()
    r = client.chat.completions.create(model="gemma4-26b", messages=msgs, max_tokens=256,
                                       temperature=0, extra_body=NO_THINK)
    return dict(answer=r.choices[0].message.content.strip(), sec=time.time() - t,
                out_tokens=r.usage.completion_tokens, in_tokens=r.usage.prompt_tokens, ctx=ctx)

facts = json.load(urllib.request.urlopen(RAW + "data/06_qlora_facts.json"))["facts"]
def hit(text, keyword_sets):
    t = text.replace(",", "").replace("，", "")
    return any(all(k in t for k in ks) for ks in keyword_sets)

UNANSWERABLE = [
    "colab-oss-lab で Llama 3.3 70B を動かしたときの速さは何トークン/秒でしたか。",
    "colab-oss-lab の実験で、A100 を使ったときの Gemma 4 31B の速さはいくつでしたか。",
    "colab-oss-lab の実験にかかった電気代はいくらですか。",
    "colab-oss-lab の実験12の結果を教えてください。",
    "colab-oss-lab で TPU を使ったときの結果はどうでしたか。",
]
REFUSE = ["記載がありません", "記載されていません", "見つかりません", "分かりません", "わかりません", "書かれていません", "載っていません"]
print(f"評価: 06 と同じ {len(facts)} 問 + 資料にない {len(UNANSWERABLE)} 問")

## 5. 評価する

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# 1) RAG なし / RAG あり（06 の確認用の聞き方）
rows = []
t_seq = time.time()
for f in facts:
    a0 = ask(f["test"], rag=False)
    a1 = ask(f["test"], rag=True)
    retrieved = any(hit(b, f["keywords"]) for _, _, b in a1["ctx"])
    rows.append(dict(id=f["id"], q=f["test"], plain=a0["answer"], plain_ok=hit(a0["answer"], f["keywords"]),
                     rag=a1["answer"], rag_ok=hit(a1["answer"], f["keywords"]), retrieved=retrieved,
                     sources=sorted({p for p, _, _ in a1["ctx"]}), rag_sec=a1["sec"], rag_in=a1["in_tokens"],
                     rag_out=a1["out_tokens"]))
n = len(rows)
plain_acc = sum(r["plain_ok"] for r in rows) / n
rag_acc = sum(r["rag_ok"] for r in rows) / n
ret_acc = sum(r["retrieved"] for r in rows) / n
print(f"RAG なし {plain_acc:.0%} / RAG あり {rag_acc:.0%} / 検索が正解の段落を拾えた {ret_acc:.0%}")

# 2) 資料にない質問
unans = []
for q in UNANSWERABLE:
    a = ask(q, rag=True)
    unans.append(dict(q=q, answer=a["answer"], refused=any(w in a["answer"] for w in REFUSE)))
refuse_n = sum(u["refused"] for u in unans)
print(f"資料にない質問で「記載がありません」: {refuse_n} / {len(unans)}")

# 3) API の速さ: 順番に 25 件 / 同時に 25 件（RAG あり）
t = time.time()
for f in facts:
    ask(f["test"], rag=True)
seq_sec = time.time() - t
t = time.time()
with ThreadPoolExecutor(max_workers=16) as ex:
    par = list(ex.map(lambda f: ask(f["test"], rag=True), facts))
par_sec = time.time() - t
out_tok = sum(p["out_tokens"] for p in par)
print(f"順番に 25 件: {seq_sec:.1f} 秒 / 同時に 25 件: {par_sec:.1f} 秒（{out_tok / par_sec:.1f} トークン/秒）")

## 6. まとめて、実行記録を出す

In [ ]:
from datetime import datetime, timezone, timedelta
vv = subprocess.check_output(["/content/vllm-env/bin/python", "-c", "import vllm; print(vllm.__version__)"], text=True).strip()
kv = re.findall(r"GPU KV cache size: ([\d,]+) tokens", open("/content/vllm_server.log").read())
checks = {
    "RAG なしは 20% 以下": plain_acc <= 0.20,
    "RAG ありは 80% 以上": rag_acc >= 0.80,
    "検索が正解の段落を拾えた割合が 80% 以上": ret_acc >= 0.80,
    "資料にない質問 5 問のうち 4 問以上で「記載がありません」": refuse_n >= 4,
    "同時に聞くと、順番に聞くより合計時間が短い": par_sec < seq_sec,
}
ok = all(checks.values())
now = datetime.now(timezone(timedelta(hours=9))).strftime("%Y-%m-%d %H:%M JST")
L = ["# 実行記録: 08 RAG・API を作る", "",
     f"- 実行日: {now}", "- 実行場所: Google Colab",
     f"- GPU: {gpu_name} / VRAM {round(vram_total_gb, 1)} GB",
     f"- モデル: {MODEL}（vLLM {vv} の OpenAI 互換 API サーバー、thinking オフ、temperature 0）",
     f"- サーバー起動: {server_start_min:.1f} 分 / KV キャッシュ: {kv[-1] if kv else '-'} トークン",
     f"- 資料: {len(FILES)} ファイル → {len(chunks)} 段落（06 の記録は除外）/ 埋め込み: intfloat/multilingual-e5-small（CPU）/ 上位 5 段落を渡す",
     f"- 想定どおりか: {'はい' if ok else 'いいえ'}", "",
     "## まとめ", "",
     "| | 正答率（06 と同じ 25 問、学習に使っていない聞き方） |", "|---|---|",
     f"| RAG なし（モデルの知識だけ） | {plain_acc:.0%} |",
     "| 参考: 06 の QLoRA（学習後） | 44% |",
     f"| **RAG あり** | **{rag_acc:.0%}** |", "",
     f"- 検索が正解の書かれた段落を拾えた割合: {ret_acc:.0%}",
     f"- 資料にない質問で「記載がありません」と答えた数: {refuse_n} / {len(unans)}",
     f"- RAG ありの 1 問あたり: 入力 {np.mean([r['rag_in'] for r in rows]):.0f} トークン / 出力 {np.mean([r['rag_out'] for r in rows]):.0f} トークン / {np.mean([r['rag_sec'] for r in rows]):.2f} 秒",
     f"- 25 問を順番に: {seq_sec:.1f} 秒 / 同時に: {par_sec:.1f} 秒（{seq_sec / par_sec:.1f} 倍速い、合計 {out_tok / par_sec:.1f} トークン/秒）", "",
     "## 判定", ""] + [f"- [{'x' if v else ' '}] {k}" for k, v in checks.items()] + ["",
     "## 問題ごと", "", "| id | 質問 | RAG なし | RAG あり | 検索 | 拾った資料 |", "|---|---|---|---|---|---|"]
for r in rows:
    L.append(f"| {r['id']} | {r['q']} | {'○' if r['plain_ok'] else '×'} | {'○' if r['rag_ok'] else '×'} | {'○' if r['retrieved'] else '×'} | {', '.join(s.split('/')[-1] for s in r['sources'])} |")
L += ["", "## 資料にない質問", "", "| 質問 | 断れた？ | 答え |", "|---|---|---|"]
for u in unans:
    L.append(f"| {u['q']} | {'○' if u['refused'] else '×'} | {u['answer'][:120].replace(chr(10), ' ')} |")
L += ["", "## 返事の例（RAG なし → RAG あり）", ""]
for r in rows:
    L += [f"### {r['q']}", "", "RAG なし:", "", "```", r["plain"][:300], "```", "", "RAG あり:", "", "```", r["rag"][:300], "```", ""]
print("\n".join(L))
json.dump(dict(rows=rows, unans=unans, seq_sec=seq_sec, par_sec=par_sec), open("/content/rag_result.json", "w"), ensure_ascii=False, indent=1)

## 7. サーバーを止める

終わったら vLLM のサーバーを止めて、**ランタイム → セッションを管理 → 解放** を押してください。

In [ ]:
server.terminate()
server.wait(timeout=60)
print("サーバーを止めました")